In [208]:
import numpy as np
import pandas as pd
from numba import njit
import vectorbt as vbt
import pandas_ta as ta
import matplotlib.pyplot as plt
import seaborn as sns

from utils.KrakenHistoricalData import KrakenHistoricalData


In [209]:
# ========= 1. Data =========
interval = '4h'
# tickers = ["BTC-USD", "ETH-USD", "DOGE-USD","LTC-USD","XRP-USD"]
# data = vbt.YFData.download(tickers, start="2025-01-01", interval=interval)
# close = data.get("Close").astype(np.float64).copy(deep=True)
# high = data.get("High").astype(np.float64).copy(deep=True)
# low = data.get("Low").astype(np.float64).copy(deep=True)

#
tickers = ["BTC", "ETH", "LTC", "DOGE", "XRP"]
k = KrakenHistoricalData()
end = pd.to_datetime("2025-09-30").tz_localize('UTC')
start = pd.to_datetime("2025-01-30").tz_localize('UTC')
data_df = k.get_ohlcv_df(tickers, interval=interval)
close = data_df.loc[start:end].xs('close', axis=1, level=1)
high = data_df.loc[start:end].xs('high', axis=1, level=1)
low = data_df.loc[start:end].xs('low', axis=1, level=1)
open = data_df.loc[start:end].xs('open', axis=1, level=1)
print(close.head())




                                     BTC          ETH         LTC      DOGE  \
2025-01-30 00:00:00+00:00  105149.898438  3200.000000  121.070000  0.332904   
2025-01-30 04:00:00+00:00  105115.296875  3191.020020  127.339996  0.331104   
2025-01-30 08:00:00+00:00  105338.601562  3223.699951  128.800003  0.332381   
2025-01-30 12:00:00+00:00  105572.000000  3255.709961  131.160004  0.336594   
2025-01-30 16:00:00+00:00  105750.203125  3272.080078  128.610001  0.334805   

                               XRP  
2025-01-30 00:00:00+00:00  3.12239  
2025-01-30 04:00:00+00:00  3.10905  
2025-01-30 08:00:00+00:00  3.10202  
2025-01-30 12:00:00+00:00  3.12122  
2025-01-30 16:00:00+00:00  3.13055  


In [210]:
# ========= 2. Signal Data Comparison =========

def entry_signals(close, high, low):
    print(f"\npandas_ta")
    sar_pandas_ta = vbt.pandas_ta('psar').run(high, low, close=close, acceleration=0.02, maximum=0.2)
    print(sar_pandas_ta.psarl.tail())
    sar_buy = sar_pandas_ta.psarl_below(close)

    adx_pandas_ta = vbt.pandas_ta('adx').run(high, low, close, length=14)
    dmp_cross = adx_pandas_ta.dmp_above(adx_pandas_ta.dmn)
    adx_threshold_cross = adx_pandas_ta.adx_above(25)
    adx_buy = dmp_cross & adx_threshold_cross
    # adx_buy.columns = buy2.columns.droplevel(0)
    print(f"\ndmp > dmn")
    print(adx_buy.tail())

    # If they're DataFrames with different column structures, extract the values
    if isinstance(sar_buy, pd.DataFrame) and isinstance(adx_buy, pd.DataFrame):
        # Align on index, then perform element-wise AND
        entries = pd.DataFrame(
            sar_buy.values & adx_buy.values,
            index=sar_buy.index,
            columns=sar_buy.columns
        )
    else:
        # If they're Series or have matching structure
        entries = sar_buy & adx_buy
    return entries


atr = vbt.pandas_ta('atr').run(high, low, close, length=14)
print(atr.atrr.tail())
atrr = atr.atrr
entries = entry_signals(close, high, low)

#exit only on ATR stop loss
exits = pd.DataFrame(False, index=entries.index, columns=entries.columns)
entry_price = close.where(entries).ffill()

# 3) 3x ATR stop distance as a fraction of entry
#    (this is already positive: 3 * atr below entry)
print(atrr.info())
print(entry_price.info())
sl_stop = (3.0 * atrr.values) / entry_price

# Optional: if entry_price is NaN (no trade yet), keep NaN
sl_stop = sl_stop.where(entry_price.notna())
print(sl_stop.info())
print(f"\nentries")
print(entries.tail())
print(entries.info())





atr_length                         14                                         
                                  BTC        ETH       LTC      DOGE       XRP
2025-09-29 08:00:00+00:00  682.510163  56.253565  1.239882  0.004392  0.041773
2025-09-29 12:00:00+00:00  797.344817  61.014732  1.310604  0.004707  0.044687
2025-09-29 16:00:00+00:00  803.812932  61.197971  1.298418  0.004667  0.044048
2025-09-29 20:00:00+00:00  770.140469  60.537381  1.267103  0.004638  0.043435
2025-09-30 00:00:00+00:00  760.151640  60.073983  1.243024  0.004587  0.041966

pandas_ta
                                     BTC          ETH         LTC      DOGE  \
2025-09-29 08:00:00+00:00  109381.096879  3912.824330  103.823492  0.224659   
2025-09-29 12:00:00+00:00  109621.401004  3922.321748  104.226273  0.225181   
2025-09-29 16:00:00+00:00  110085.150747  3939.455236  104.697194  0.225683   
2025-09-29 20:00:00+00:00  110602.800470  3955.560715  105.102187  0.226165   
2025-09-30 00:00:00+00:00  111136.284186 

In [211]:
from vectorbt.portfolio import StopEntryPrice

pf = vbt.Portfolio.from_signals(close,
                                entries=entries,
                                exits=exits,
                                size=1,
                                size_type='percent',
                                direction='longonly',
                                sl_stop=sl_stop,
                                sl_trail=True,
                                freq=interval,
                                fees=0.004,
                                init_cash=1000,
                                stop_entry_price=StopEntryPrice.Close
                                )

In [212]:
pf_btc = pf[pf.wrapper.columns[0]]

stats_df = pd.DataFrame()
for col in pf.wrapper.columns:
    pf_col = pf[col]
    stats_df = pd.concat([stats_df, pf_col.stats()], axis=1)

print(stats_df.to_string())

# Stop price recorded at trade execution level
trade_records = pf.trades.records_readable
# print(trade_records.info())

print(trade_records[['Column', 'Avg Entry Price', 'Avg Exit Price', 'PnL', 'Return', 'Status']])

                                                  BTC                        ETH                        LTC                       DOGE                        XRP
Start                       2025-01-30 00:00:00+00:00  2025-01-30 00:00:00+00:00  2025-01-30 00:00:00+00:00  2025-01-30 00:00:00+00:00  2025-01-30 00:00:00+00:00
End                         2025-09-30 00:00:00+00:00  2025-09-30 00:00:00+00:00  2025-09-30 00:00:00+00:00  2025-09-30 00:00:00+00:00  2025-09-30 00:00:00+00:00
Period                              243 days 04:00:00          243 days 04:00:00          243 days 04:00:00          243 days 04:00:00          243 days 04:00:00
Start Value                                    1000.0                     1000.0                     1000.0                     1000.0                     1000.0
End Value                                 1050.926912                1740.690254                 759.970826                1329.240483                1059.261768
Total Return [%]            

In [216]:
n = 1
pf[pf.wrapper.columns[n]].plot(width=1200, height=1900, title=f"{tickers[n]}").show()